# **Ascend C算子性能优化典型示例**

本节为算子性能优化典型示例介绍章节，完成本章节内容的学习可以掌握如何优化已开发好的算子。我们将按照以下结构，带你学习Ascend C算子性能优化：
- 环境准备
- Tiling策略优化
- 流水编排优化
- 内存访问优化
- 课后实践
---

## **1 环境准备**
本文所有内容均存放于Sources文件夹。
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。保证能正常导入代码以及正常使用工具。

In [ ]:
!mkdir -p Sources/08.04

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

---
## **2 Tiling策略优化**
AI处理器的物理核数是固定的，当数据切分之后，可能发生部分核有计算拖尾的情况，即每次所有核计算量除以每个核处理的数据量不能被核数整除，导致最后需要部分尾核来计算尾块数据。而在尾核计算时，部分核始终处于空闲状态，从而使得算子的整体性能变差。这里我们模拟一下分配不均匀的场景，假设总的数据量为[45, 20480]，原Tiling设计每个核计算20480个数据，使用40个Vector核(这里假设设备有40个Vector核可用)，多余的数据由前5个核每个核多处理20480个数据。  

<img src="./images/tiling_error.png" alt="tiling"  width="700px">  

假设每个核处理20480数据的耗时相同为x，那么这种方案的算子耗时为2x。

### **2.1 获取原始算子性能数据**
尝试运行已提供的算子不均分实现调用并抓取[45, 20480]的half类型输入时的性能数据。

In [ ]:
# 清理已存在的custom_op
!rm -rf Sources/08.04/custom_op

# 复制之前课程的算子工程, 并修改算子工程内CANN包路径配置为实际路径
!cp -r src/custom_op_04_01 Sources/08.04/custom_op;sed -i '/"ASCEND_CANN_PACKAGE_PATH": {/,/}/ s|\s*"value": ".*"| "value": "'"$ASCEND_TOOLKIT_HOME"'"|' Sources/08.04/custom_op/CMakePresets.json

# 编译部署算子
!cd Sources/08.04/custom_op;bash build.sh;./build_out/custom_opp*.run  --install-path=${HOME}/

# 清除已有的测试代码
!rm -rf Sources/08.04/test;

# 复制已有的测试代码
!cp -r src/custom_op_04_01/test Sources/08.04/test;

# 编译调用代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/08.04/test/main.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/08.04/execute_op;

!rm -rf Sources/08.04/prof
# 创建性能文件存放目录
!mkdir -p Sources/08.04/prof
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;msprof op --output=Sources/08.04/prof ./Sources/08.04/execute_op


查看抓取到的耗时数据

In [ ]:
import pandas as pd
import glob

csv_file = glob.glob("Sources/08.04/prof/*/OpBasicInfo.csv")[0]
df = pd.read_csv(csv_file)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df

针对上述切分策略，调整拖尾核的位置后可以达到全局负载最优，即所有核的计算量基本相等。每个核多处理2560数据，总耗时为x + 2560数据耗时， 2560 < 20480, 所以处理耗时小于x， 整体耗时预期小于2x。

<img src="./images/tiling_right.png" alt="tiling"  width="700px">    

尝试基于已提供优化思路修改代码。

### **2.2 算子性能优化**
#### **2.2.1 Host侧修改**  
修改op_host/add_custom_template.cpp文件内的Tiling实现函数，假设我们有40个可用的Vector核时，设置为45会把数据均分为45份，先把前40份数据分配给每个核，然后把剩余的5份数据分配给前5个处理完数据的核，与我们模拟的场景一致。**这里可以修改SetBlockDim设置核数为40，每个核处理23040个数据，数据均分，不会有核会多处理一块数据**。

In [ ]:
%%writefile Sources/08.04/custom_op/op_host/add_custom_template.cpp 
#include "../op_kernel/add_custom_template_tiling.h"
#include "register/op_def_registry.h"

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{
  AddCustomTemplateTilingData *tiling = context->GetTilingData<AddCustomTemplateTilingData>();
  uint32_t totalLength = context->GetInputShape(0)->GetOriginShape().GetShapeSize();
  context->SetBlockDim(45);
  tiling->totalLength = totalLength;
  tiling->tileNum = 1;
  return ge::GRAPH_SUCCESS;
}
}

namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}

namespace ops {
class AddCustomTemplate : public OpDef {
public:
    explicit AddCustomTemplate(const char* name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("z")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");
    }
};
OP_ADD(AddCustomTemplate);
}

#### **2.2.2 Tiling结构体定义**  
该部分可以不修改

In [ ]:
%%writefile Sources/08.04/custom_op/op_kernel/add_custom_template_tiling.h
#ifndef ADD_CUSTOM_TEMPLATE_TILING_H
#define ADD_CUSTOM_TEMPLATE_TILING_H
#include <cstdint>

struct AddCustomTemplateTilingData {
    uint32_t totalLength;
    uint32_t tileNum;
};
#endif 

#### **2.2.3 Kernel侧实现**  
该部分可不修改，代码中printf为增加单个核执行耗时的操作，以便于更直观的观察到性能优化。

In [ ]:
%%writefile Sources/08.04/custom_op/op_kernel/add_custom_template.cpp 
#include "kernel_operator.h"
#include "add_custom_template_tiling.h"
constexpr int32_t BUFFER_NUM = 1;  // tensor num for each queue

template <class dtypeX, class dtypeY, class dtypeZ>
class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
        xGm.SetGlobalBuffer((__gm__ dtypeX *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        yGm.SetGlobalBuffer((__gm__ dtypeY *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        zGm.SetGlobalBuffer((__gm__ dtypeZ *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(dtypeX));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(dtypeY));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(dtypeZ));
    }

    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
        AscendC::printf("Core %d executed %d times in total\n",  AscendC::GetBlockIdx(), loopCount);
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.AllocTensor<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.AllocTensor<dtypeY>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.DeQue<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.DeQue<dtypeY>();
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.AllocTensor<dtypeZ>();
        AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
        outQueueZ.EnQue<dtypeZ>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.DeQue<dtypeZ>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<dtypeX> xGm;
    AscendC::GlobalTensor<dtypeY> yGm;
    AscendC::GlobalTensor<dtypeZ> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

 __global__ __aicore__ void add_custom_template(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling)
{
    REGISTER_TILING_DEFAULT(AddCustomTemplateTilingData);
    GET_TILING_DATA_WITH_STRUCT(AddCustomTemplateTilingData, tiling_data, tiling);
    KernelAdd<DTYPE_X, DTYPE_Y, DTYPE_Z> op;
    op.Init(x, y, z, tiling_data.totalLength, tiling_data.tileNum);
    op.Process();
}

修改后重新部署算子抓取性能数据

In [ ]:
# 编译部署算子
!cd Sources/08.04/custom_op;bash build.sh;./build_out/custom_opp*.run  --install-path=${HOME}/

# 清除可能存在的性能文件
!rm -rf Sources/08.04/prof2
# 创建性能文件存放目录
!mkdir -p Sources/08.04/prof2
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;msprof op --output=Sources/08.04/prof2 ./Sources/08.04/execute_op

抓取性能后，读取性能数据与之前对比，观察是否符合预期，减少了耗时。

In [ ]:
import pandas as pd
import glob

csv_file = glob.glob("Sources/08.04/prof2/*/OpBasicInfo.csv")[0]
df = pd.read_csv(csv_file)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df

---
## **3 流水编排优化**
执行于AI Core上的指令队列主要包括如下几类，Vector指令队列（V）、Cube指令队列（M）、Scalar指令队列（S）和搬运指令队列（MTE1/MTE2/MTE3）。以AddCustomTemplate算子为例，矢量计算前后的CopyIn、CopyOut过程使用搬运指令队列（MTE2/MTE3），Compute过程使用Vector指令队列（V），不同指令队列可并行执行，意味着CopyIn、CopyOut过程和Compute过程是可以并行的。一个完整的数据搬运和计算过程，CopyIn过程将数据从Global Memory搬运到Local Memory，Vector计算单元完成Compute计算后，经过CopyOut过程将计算结果搬回Global Memory。 

<img src="./images/single_buffer.png" alt="buffer"  width="350px">  

在此过程中，数据搬运与Vector计算串行执行，Vector计算单元不可避免存在资源闲置问题，假设CopyIn、Compute、CopyOut三阶段分别耗时相同均为t，则Vector的利用率仅为1/3，等待时间过长，Vector利用率严重不足。

### **3.1 获取原始的算子性能数据**
这里我们仍以准备好的AddCustomTemplate算子为例，尝试运行已提供不开启DoubleBuffer的算子实现调用并抓取[45, 20480]的half类型输入时的性能数据。

In [ ]:
# 清理已存在的custom_op
!rm -rf Sources/08.04/custom_op

# 复制之前课程的算子工程, 并修改算子工程内CANN包路径配置为实际路径
!cp -r src/custom_op_04_02 Sources/08.04/custom_op;sed -i '/"ASCEND_CANN_PACKAGE_PATH": {/,/}/ s|\s*"value": ".*"| "value": "'"$ASCEND_TOOLKIT_HOME"'"|' Sources/08.04/custom_op/CMakePresets.json

# 编译部署算子
!cd Sources/08.04/custom_op;bash build.sh;./build_out/custom_opp*.run  --install-path=${HOME}/

# 清除已有的测试代码
!rm -rf Sources/08.04/test;

# 复制已有的测试代码
!cp -r src/custom_op_04_02/test Sources/08.04/test;

# 编译调用代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/08.04/test/main.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/08.04/execute_op;

!rm -rf Sources/08.04/prof
# 创建性能文件存放目录
!mkdir -p Sources/08.04/prof
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;msprof op --output=Sources/08.04/prof ./Sources/08.04/execute_op


查看不开启DoubleBuffer时的性能数据

In [ ]:
import pandas as pd
import glob

csv_file = glob.glob("Sources/08.04/prof/*/OpBasicInfo.csv")[0]
df = pd.read_csv(csv_file)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df

为减少Vector等待时间，使能DoubleBuffer机制将待处理的数据一分为二，当Vector单元对Tensor1中数据进行Compute计算时，Tensor2数据流可以执行CopyIn的过程；而当Vector切换到计算Tensor2时，Tensor1数据流可以执行CopyOut的过程。由此，数据的进出搬运和Vector计算实现并行执行，Vector闲置问题得以有效缓解。   
 
<img src="./images/double_buffer.png" alt="buffer"  width="350px">    

尝试基于已提供优化思路修改代码。

### **3.2 算子性能优化**
#### **3.2.1 Host侧代码实现**  
不需要修改

In [ ]:
%%writefile Sources/08.04/custom_op/op_host/add_custom_template.cpp 
#include "../op_kernel/add_custom_template_tiling.h"
#include "register/op_def_registry.h"

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{
  AddCustomTemplateTilingData *tiling = context->GetTilingData<AddCustomTemplateTilingData>();
  uint32_t totalLength = context->GetInputShape(0)->GetOriginShape().GetShapeSize();
  context->SetBlockDim(40);
  tiling->totalLength = totalLength;
  tiling->tileNum = 1;
  return ge::GRAPH_SUCCESS;
}
}

namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}

namespace ops {
class AddCustomTemplate : public OpDef {
public:
    explicit AddCustomTemplate(const char* name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("z")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");
    }
};
OP_ADD(AddCustomTemplate);
}

#### **3.2.2 Tiling结构体定义**  
不需要修改

In [ ]:
%%writefile Sources/08.04/custom_op/op_kernel/add_custom_template_tiling.h
#ifndef ADD_CUSTOM_TEMPLATE_TILING_H
#define ADD_CUSTOM_TEMPLATE_TILING_H
#include <cstdint>

struct AddCustomTemplateTilingData {
    uint32_t totalLength;
    uint32_t tileNum;
};
#endif 

#### **3.2.3 Kernel侧实现**  
**修改BUFFER_NUM为2，开启DoubleBuffer机制**。Compute内的循环50次是为了增加计算量，模拟Vector被占用。

In [ ]:
%%writefile Sources/08.04/custom_op/op_kernel/add_custom_template.cpp 
#include "kernel_operator.h"
#include "add_custom_template_tiling.h"
constexpr int32_t BUFFER_NUM = 1;  // tensor num for each queue

template <class dtypeX, class dtypeY, class dtypeZ>
class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
        xGm.SetGlobalBuffer((__gm__ dtypeX *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        yGm.SetGlobalBuffer((__gm__ dtypeY *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        zGm.SetGlobalBuffer((__gm__ dtypeZ *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(dtypeX));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(dtypeY));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(dtypeZ));
    }

    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.AllocTensor<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.AllocTensor<dtypeY>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.DeQue<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.DeQue<dtypeY>();
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.AllocTensor<dtypeZ>();
        for (int32_t j = 0; j < 50; j++) {
            AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
        }
        outQueueZ.EnQue<dtypeZ>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.DeQue<dtypeZ>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<dtypeX> xGm;
    AscendC::GlobalTensor<dtypeY> yGm;
    AscendC::GlobalTensor<dtypeZ> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

 __global__ __aicore__ void add_custom_template(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling)
{
    REGISTER_TILING_DEFAULT(AddCustomTemplateTilingData);
    GET_TILING_DATA_WITH_STRUCT(AddCustomTemplateTilingData, tiling_data, tiling);
    KernelAdd<DTYPE_X, DTYPE_Y, DTYPE_Z> op;
    op.Init(x, y, z, tiling_data.totalLength, tiling_data.tileNum);
    op.Process();
}

编译部署修改后的算子，并重新抓取性能数据。

In [ ]:
# 编译部署算子
!cd Sources/08.04/custom_op;bash build.sh;./build_out/custom_opp*.run  --install-path=${HOME}/

# 清除可能存在的性能文件
!rm -rf Sources/08.04/prof2
# 创建性能文件存放目录
!mkdir -p Sources/08.04/prof2
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;msprof op --output=Sources/08.04/prof2 ./Sources/08.04/execute_op

查看开启DoubleBuffer机制后的性能数据，观察性能是否有提升。

In [ ]:
import pandas as pd
import glob

csv_file = glob.glob("Sources/08.04/prof2/*/OpBasicInfo.csv")[0]
df = pd.read_csv(csv_file)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df

## **4 内存访问优化**
搬运不同大小的数据块时，对带宽的利用率不一样。根据实测经验，单次搬运数据长度16KB以上时，通常能较好地发挥出带宽的最佳性能。因此对于单次搬运，应考虑尽可能的搬运较大的数据块。

### **4.1 获取原始算子性能数据**
这里我们仍以准备好的AddCustomTemplate算子为例，尝试运行已提供每次搬运128B的算子实现调用并抓取[45, 20480]的half类型输入时的性能数据。

In [ ]:
# 清理已存在的custom_op
!rm -rf Sources/08.04/custom_op

# 复制之前课程的算子工程, 并修改算子工程内CANN包路径配置为实际路径
!cp -r src/custom_op_04_03 Sources/08.04/custom_op;sed -i '/"ASCEND_CANN_PACKAGE_PATH": {/,/}/ s|\s*"value": ".*"| "value": "'"$ASCEND_TOOLKIT_HOME"'"|' Sources/08.04/custom_op/CMakePresets.json

# 编译部署算子
!cd Sources/08.04/custom_op;bash build.sh;./build_out/custom_opp*.run  --install-path=${HOME}/

# 清除已有的测试代码
!rm -rf Sources/08.04/test;

# 复制已有的测试代码
!cp -r src/custom_op_04_03/test Sources/08.04/test;

# 编译调用代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/08.04/test/main.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/08.04/execute_op;

!rm -rf Sources/08.04/prof
# 创建性能文件存放目录
!mkdir -p Sources/08.04/prof
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;msprof op --output=Sources/08.04/prof ./Sources/08.04/execute_op


查看每次搬运128B时的性能数据

In [ ]:
import pandas as pd
import glob

csv_file = glob.glob("Sources/08.04/prof/*/OpBasicInfo.csv")[0]
df = pd.read_csv(csv_file)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df

根据我们单次搬运数据长度16KB以上时通常能较好地发挥出带宽的最佳性能的经验，尝试修改代码。

### **4.2 算子性能优化**
#### **4.2.1 Host侧修改**  
每次的搬运量为 45 * 20480 * sizeof(half) / tileNum，**修改tileNum为1或其他合适的值，使每次搬运的数据在16KB以上**。

In [ ]:
%%writefile Sources/08.04/custom_op/op_host/add_custom_template.cpp 
#include "../op_kernel/add_custom_template_tiling.h"
#include "register/op_def_registry.h"

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{
  AddCustomTemplateTilingData *tiling = context->GetTilingData<AddCustomTemplateTilingData>();
  uint32_t totalLength = context->GetInputShape(0)->GetOriginShape().GetShapeSize();
  context->SetBlockDim(40);
  tiling->totalLength = totalLength;
  tiling->tileNum = 360;
  return ge::GRAPH_SUCCESS;
}
}

namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}

namespace ops {
class AddCustomTemplate : public OpDef {
public:
    explicit AddCustomTemplate(const char* name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("z")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");
    }
};
OP_ADD(AddCustomTemplate);
}

#### **4.2.2 Tiling结构体定义**  
不需要修改

In [ ]:
%%writefile Sources/08.04/custom_op/op_kernel/add_custom_template_tiling.h
#ifndef ADD_CUSTOM_TEMPLATE_TILING_H
#define ADD_CUSTOM_TEMPLATE_TILING_H
#include <cstdint>

struct AddCustomTemplateTilingData {
    uint32_t totalLength;
    uint32_t tileNum;
};
#endif 

#### **4.2.3 Kernel侧实现**  
不需要修改  

In [ ]:
%%writefile Sources/08.04/custom_op/op_kernel/add_custom_template.cpp 
#include "kernel_operator.h"
#include "add_custom_template_tiling.h"
constexpr int32_t BUFFER_NUM = 1;  // tensor num for each queue

template <class dtypeX, class dtypeY, class dtypeZ>
class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
        xGm.SetGlobalBuffer((__gm__ dtypeX *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        yGm.SetGlobalBuffer((__gm__ dtypeY *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        zGm.SetGlobalBuffer((__gm__ dtypeZ *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(dtypeX));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(dtypeY));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(dtypeZ));
    }

    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.AllocTensor<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.AllocTensor<dtypeY>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.DeQue<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.DeQue<dtypeY>();
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.AllocTensor<dtypeZ>();
        AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
        outQueueZ.EnQue<dtypeZ>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.DeQue<dtypeZ>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<dtypeX> xGm;
    AscendC::GlobalTensor<dtypeY> yGm;
    AscendC::GlobalTensor<dtypeZ> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

 __global__ __aicore__ void add_custom_template(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling)
{
    REGISTER_TILING_DEFAULT(AddCustomTemplateTilingData);
    GET_TILING_DATA_WITH_STRUCT(AddCustomTemplateTilingData, tiling_data, tiling);
    KernelAdd<DTYPE_X, DTYPE_Y, DTYPE_Z> op;
    op.Init(x, y, z, tiling_data.totalLength, tiling_data.tileNum);
    op.Process();
}

编译部署修改后的代码，并抓取性能数据

In [ ]:
# 编译部署算子
!cd Sources/08.04/custom_op;bash build.sh;./build_out/custom_opp*.run  --install-path=${HOME}/

# 清除可能存在的性能文件
!rm -rf Sources/08.04/prof2
# 创建性能文件存放目录
!mkdir -p Sources/08.04/prof2
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;msprof op --output=Sources/08.04/prof2 ./Sources/08.04/execute_op

查看搬运数据量在16KB以上时，是否如预期降低了算子耗时。

In [ ]:
import pandas as pd
import glob

csv_file = glob.glob("Sources/08.04/prof2/*/OpBasicInfo.csv")[0]
df = pd.read_csv(csv_file)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df

---
## **课后实践**

请根据本节课程学习内容完成以下题目进行自测。

1. （单选题）下列哪个操作不能优化算子性能？  
    A. 通过尽可能搬运512Byte对齐数据提升Global Memory搬运到Local Memory的效率  
    B. 数据量比较大时，开启Double Buffer，尽可能提高硬件利用率。  
    C. 数据量比较小时，使用合适的核数，避免数据分块导致的额外开销  
    D. Tiling设计时尽可能切分多次，加快Vector核运算速度  

2. （单选题）算子性能瓶颈最常见的原因之一是：  
    A. 代码注释不够详细  
    B. 数据搬运与计算没有充分并行  
    C. 算子输出精度过高  
    D. 编译选项过于复杂  

3. （单选题）优化Tiling设计时，可以达到性能优化目的的操作是：  
    A. 减少整体运算数据量，加快计算速度  
    B. 合理切分数据，提升计算与搬运的并行度  
    C. 减少算子输入输出张量数量  
    D. Tiling设计时尽可能使用最大可能核数，加速计算  

4. （单选题）观察算子性能、定位瓶颈最直接的方式是：  
    A. 查看仿真耗时数据或msProf工具抓取的耗时、利用率数据  
    B. 凭经验猜测  
    C. 观察代码行数  
    D. 对比不同编译器版本    

5. （单选题）在 AscendC 算子性能优化流程中，正确的顺序是：  
    A. 写代码 → 直接上线 → 再看性能  
    B. 实现功能 → 采集性能基线 → 定位瓶颈 → 迭代优化 → 验证  
    C. 先优化再实现功能  
    D. 只做功能验证，不做性能优化  

执行以下代码查看答案：

In [ ]:
!cat ./answer/08.04_answer.txt